**GEOG5415M Programming for Spatial Data Science**

# Week 9: Reproducible spatial data science
## Can you see deprivation from the street?

In this practical you will work with data from a real research project. The research asks a simple question: **how much can photographs of a neighbourhood's streets tell us about the average socio-economic status of the people who live there?**

The method was first developed for Greater Manchester ([Asher et al., 2027](https://doi.org/10.1016/j.compenvurbsys.2026.102526)). In Manchester, a model that used only street-view photographs explained about 68% of the variation in neighbourhood-level variation (using the Index of Multiple Deprivation). Google Street View images cannot be shared, so the work is currently being repeated for **Amsterdam**, where the City publishes its own street-level photographs under an open licence.

### What has already been done for you

Some of the steps in the research take many hours and need a powerful computer, so we have done them for you:

1. About 30,000 locations were sampled along the streets of Amsterdam. At each location, four photographs were taken from the City's 360° panoramas (facing north, east, south and west). That is about 120,000 photographs.
2. A vision transfomer (a type of foundation model) called **CLIP** turned each photograph into a list of 512 numbers, called an **embedding**. Photographs that look similar have similar embeddings.
3. The results were saved in a few small files, which you will download from the course repository.

All of the code that did this is openly available in the [StreetViewDeprivation-Amsterdam repository](https://github.com/nickmalleson/StreetViewDeprivation-Amsterdam). That is the point of reproducible research: you do not need to repeat every step, but anyone *could*, because the code, the data sources and the processing steps are all published.

### What you will do

1. Record the software versions and set a random seed (Section 1)
2. Load the data and check it has loaded correctly with `assert` statements (Section 2)
3. Look at photographs from two very different parts of Amsterdam (Section 3)
4. Ask CLIP which streets look like a description, and map the results (Section 4)
5. Calculate median embeddings for a few buurten, then predict socio-economic status with a random forest, as in Week 7, using the embedding dimensions as the explanatory variables (Section 5)
6. See how the results change when you change the random seed (Section 6)
7. Check that your notebook is reproducible and save it to GitHub (Section 7)

The optional extensions after that are for anyone who wants to go further.

As in previous weeks, <b><font color='orchid'>instructions and tasks for you to complete are in purple</font></b>.

### Before you start

If you followed the instructions in the lecture, you should have a GitHub repository called `GEOG5415`. Save a copy of this notebook there now: **File → Save a copy in GitHub**, choose your `GEOG5415` repository, and write a commit message such as *"Add week 9 notebook"*.

## 1. Setting up

This section uses three habits that make a notebook easier to reproduce:

1. **Record the versions of the libraries you use.** Libraries change. If someone gets a different result from your notebook next year, this is the first thing they will want to check.
2. **Set a random seed.** Many data science methods use random numbers (for example, to split data into training and test sets). Setting the seed means you get the same 'random' numbers every time.
3. **Keep file locations in one place.** All of the data in this notebook are read from one address, `DATA_URL`. If the data move, you only need to change one line.

In [1]:
# Libraries for working with data and maps
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning libraries (the same ones as in Week 7)
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
# Record when the notebook was run and which versions of Python and the libraries were used
import sys
import datetime

print("Notebook run on:", datetime.date.today())
print("Python:", sys.version.split()[0])
for library in [pd, gpd, np, matplotlib, sns, sklearn]:
    print(library.__name__, library.__version__)

Copy these versions into the README of your repository when you write it in Section 7. They tell other people (and future you) which versions the notebook is known to work with.

In [ ]:
# The random seed. Any whole number will do; 42 is a common choice.
RANDOM_STATE = 42

# Where the data are stored (the week_9 folder of the course repository)
DATA_URL = "https://github.com/MSc-Urban-Environmental-Leeds/GEOG5415M-Programming-for-Spatial-Data-Science/raw/refs/heads/main/data/week_9/"

The following chunk has 'helper' functions that we will use in the rest of the notebook

In [ ]:
#@title Helper functions for showing photographs (run this cell; you do not need to read the code) { display-mode: "form" }
import io
import urllib.request
from PIL import Image

def open_image(file_name, folder="images/"):
    """Open one of the week 9 photographs from the course repository (or a local folder)."""
    location = DATA_URL + folder + file_name
    if location.startswith("http"):
        with urllib.request.urlopen(location) as response:
            return Image.open(io.BytesIO(response.read()))
    return Image.open(location)

def show_images(image_rows, titles, ncols=6):
    """Show the photographs listed in image_rows (some rows of the images table), with titles."""
    nrows = (len(image_rows) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(2.6 * ncols, 2.9 * nrows), squeeze=False)
    for ax in axes.flat:
        ax.set_axis_off()
    for ax, file_name, title in zip(axes.flat, image_rows['image_file'], titles):
        ax.imshow(open_image(file_name))
        ax.set_title(title, fontsize=9)
    plt.tight_layout()
    plt.show()

def show_buurt_images(images, buurtcode):
    """Show the six photographs of one buurt. The titles show when each photograph was taken."""
    rows = images.loc[(images['purpose'] == 'buurt') & (images['buurtcode'] == buurtcode)]
    if len(rows) == 0:
        print("There are no photographs for", buurtcode)
        return
    print(rows['buurt_name'].iloc[0], "(" + buurtcode + ")")
    show_images(rows, titles=rows['image_date'])

def show_prompt_images(images, prompt):
    """Show the four photographs in Amsterdam that best match one of the descriptions."""
    rows = images.loc[(images['purpose'] == 'prompt') & (images['prompt'] == prompt)].sort_values('rank')
    if len(rows) == 0:
        print("There is no description called", prompt)
        return
    print("Best matches for:", prompt)
    show_images(rows, titles=rows['buurt_name'], ncols=4)

## 2. Load and check the data

The data come from four places. All of them are published under an open licence (CC-BY 4.0), which means we can share and reuse them as long as we say where they came from. The `README.md` file in the [week_9 data folder](https://github.com/MSc-Urban-Environmental-Leeds/GEOG5415M-Programming-for-Spatial-Data-Science/tree/main/data/week_9) records exactly how each file was made. This is the kind of information your own project README should include.

| File | Contents | Source |
|------|----------|--------|
| `amsterdam_buurten_2024.geojson` | Boundaries of Amsterdam's 517 *buurten* (neighbourhoods) | Statistics Netherlands (CBS) |
| `amsterdam_ses_2023.csv` | Socio-economic status score (SES-WOA) for each buurt in 2023 | CBS |
| `amsterdam_street_points.csv` | The 30,155 photograph locations, with a CLIP score for 15 descriptions | Gemeente Amsterdam (photographs), CLIP (scores) |
| `prompts.csv` | The 15 descriptions | - |
| `images.csv` and `images/` | 107 of the photographs, for you to look at | Gemeente Amsterdam |

The **SES-WOA score** combines the wealth, education level and employment history of the households in each buurt. Higher scores mean higher socio-economic status. It is the Dutch equivalent of the deprivation indices you have met for the UK, but note that the scale is reversed: a *low* SES score means a *more deprived* area.

In [ ]:
buurten = gpd.read_file(DATA_URL + "amsterdam_buurten_2024.geojson")
ses = pd.read_csv(DATA_URL + "amsterdam_ses_2023.csv")
points = pd.read_csv(DATA_URL + "amsterdam_street_points.csv")
prompts = pd.read_csv(DATA_URL + "prompts.csv")
images = pd.read_csv(DATA_URL + "images.csv")

In [ ]:
buurten.head()

In [ ]:
ses.head()

In [ ]:
points.head()

In [ ]:
prompts.head()

In [ ]:
images.head()

### Explore the data

Before you go any further, explore the data in the same way as in previous weeks. <font color='orchid'> <b>Use the cells below to explore <code>buurten</code>, <code>ses</code> and <code>points</code>.</b></font> For example:

- How many rows and columns does each one have?
- What type of data is stored in each column? Are the types what you would expect?
- Are there any missing values?
- Do the summary statistics (minimum, maximum, mean) look sensible?

You may have noticed that `points` is an ordinary pandas data frame, not a GeoDataFrame. It has latitude and longitude columns but no geometry, so you cannot map it yet. You will turn it into a GeoDataFrame in Section 4.

In [ ]:
# Add your code here



In [ ]:
# Add your code here



### Checking the data with `assert`

In Week 3 you checked data by looking at them (`.head()`, `.describe()`, `.isna().sum()`). That works, but you have to remember to look, and you have to notice when something is wrong.

An `assert` statement does the checking for you. It tests a condition. If the condition is `True`, nothing happens. If it is `False`, Python stops with an error and prints your message:

```python
assert condition, "message to show if the condition is False"
```

Checks like these are a simple form of **software testing**. If the data change (for example, a publisher updates a file), your notebook will stop at the check instead of quietly producing different results.

In [ ]:
assert buurten['buurtcode'].is_unique, "Each buurt should appear only once"
assert len(buurten) == 517, "Expected 517 buurten in Amsterdam"
assert buurten.crs.to_epsg() == 4326, "Expected latitude and longitude (EPSG:4326)"
print("All checks passed")

<font color='orchid'> <b>Write <code>assert</code> statements to check that:</b></font>

1. `ses` has 420 rows
2. there are no missing values in the `ses_score` column
3. every `buurtcode` in `ses` also appears in `buurten`

*Hint:* for the last one, `ses['buurtcode'].isin(buurten['buurtcode'])` gives `True` or `False` for each row, and `.all()` tells you whether they are all `True`. If you need help to understand this one please ask.

In [ ]:
# Add your code here



### When were the photographs taken?

Always read the metadata. The `image_date` column records the year and month each photograph was taken. The `pano_id` column records the City of Amsterdam's own identifier for the panorama that each photograph came from. This links our processed data back to the original source, so anyone can find the photograph in the City's data.

In [ ]:
points['image_year'] = points['image_date'].str[:4]  # The first four characters are the year
points['image_year'].value_counts().sort_index()

Most photographs were taken in 2025, but some are almost ten years older, because not every street is photographed every year. The SES scores are for 2023. Neighbourhoods change, so this mismatch is a limitation that a research paper should mention.

Data can also disappear. The City of Amsterdam has **withdrawn the photographs from its 2023 survey**: the records are still listed, but the images can no longer be downloaded. (The 56 photographs dated 2023 above were taken in January to March 2023 as part of the City's 2022 survey.) In neighbouring Almere, where 2023 was the most recent survey, the research code had to use photographs from 2017 to 2020 instead. Data that you download today may not be available tomorrow, which is one reason to record what you downloaded and when.

### Join the SES scores to the boundaries and map them

In [ ]:
buurten_ses = buurten.merge(ses, on='buurtcode', how='left')
print("Buurten without an SES score:", buurten_ses['ses_score'].isna().sum())

In [ ]:
# How many people live in the buurten without a score?
buurten_ses.loc[buurten_ses['ses_score'].isna(), 'population'].describe()

97 buurten have no SES score. Half of them have 15 or fewer residents: they are mostly parks, harbours and industrial areas. As in Week 8, ask who (and where) is missing from your data.

<font color='orchid'> <b>Make a map of <code>ses_score</code> using <code>buurten_ses</code> (see Week 5).</b></font> Use a diverging colour map such as `'RdBu'`, so that low and high scores have different colours, and set `vmin=-1, vmax=1` so that a score of 0 is in the middle of the colour scale. Show the buurten without a score in light grey.

In [ ]:
# Add your code here



## 3. Look at the streets

Before asking a computer, try it yourself. The `images` table includes six photographs from each of eight buurten in two very different parts of Amsterdam:

- **Centrum** is the historic city centre, with canals and buildings from the 17th to the 19th centuries.
- **Zuidoost** is a district built mostly from the late 1960s onwards, including the high-rise Bijlmermeer estate.

In [ ]:
buurt_list = images.loc[images['purpose'] == 'buurt', ['area', 'buurtcode', 'buurt_name']].drop_duplicates()
buurt_list

One of the `helper` functions defined at the start is `show_buurt_images()` which we can use to conveniently show images using a buurtcode:

In [ ]:
show_buurt_images(images, 'BU0363AG04')

<font color='orchid'> <b>Use <code>show_buurt_images()</code> to look at the photographs from at least two buurten in Centrum and two in Zuidoost.</b></font> For each area, which buurt do you think has the highest SES score, and which the lowest? <font color='orchid'> <b>Write down your guesses, then run the cell after next to check them.</b></font>

In [ ]:
# Add your code here



In [ ]:
buurt_list.merge(ses[['buurtcode', 'ses_score']], on='buurtcode').sort_values(['area', 'ses_score'])

How well did you do? What did you look at to make your guesses? Some of the differences are obvious (canals or high-rise flats), but within each area the differences are much harder to see.

## 4. Asking CLIP about the streets

In Section 5 the model will use all 512 numbers in each embedding. Those numbers mean nothing to us, so first we use CLIP to turn them into something we can understand: how closely each street matches a short text description. This lets us check what CLIP 'sees', and where in Amsterdam the streets that match each description are.

CLIP is a *vision-language* model. It was trained on hundreds of millions of photographs and their captions, so it can turn **both** photographs and text into embeddings of 512 numbers. If a photograph matches a description, their embeddings will be similar.

We gave CLIP the 15 descriptions in `prompts.csv` (for example, *"a photo of a canal lined with old houses"*) and compared each one with every photograph. The result is a **CLIP score**: a number that says how closely the photographs at a location match a description. Each description has its own score column in `points`; for example, the `canal` column holds the scores for the canal description.

The scores have been standardised, so they are easy to compare:

- **0** means the location matches the description about as well as an average Amsterdam location
- **2** means it matches much better than average (for `canal`, it probably is next to a canal)
- **-2** means it matches much worse than average

In [ ]:
# These are the different descriptions we have made up
prompts

Is CLIP finding what we asked for? `show_prompt_images()` shows the four best-matching photographs in Amsterdam for a description.

In [ ]:
show_prompt_images(images, 'canal')

<font color='orchid'> <b>Look at the best matches for two other descriptions.</b></font> Does CLIP find what you expected? Are there any surprises?

In [ ]:
# Add your code here



### Check that the locations and the buurten line up

To map the scores we need `points` as a GeoDataFrame (see Week 5). Whenever you use two spatial datasets together, first check that they line up: that they use the same CRS, and that the features are where you expect them to be. Errors here are common and easy to miss. If one layer uses a different CRS, or has latitude and longitude the wrong way round, a spatial join may quietly match nothing, or the wrong things.

The code below does three things:

1. Turns the `points` table into a GeoDataFrame
2. Checks with `assert` that both layers use the same CRS
3. Uses a spatial join to check that every photograph location falls inside a buurt. If the layers did not line up, many locations would fall outside the buurten and the `assert` would fail

In [ ]:
# Turn the points dataframe into a geodataframe, using the lat/lon columns
points_gdf = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['longitude'], points['latitude']),
    crs='EPSG:4326'
)

# Check that the two layers use the same coordinate reference system
assert points_gdf.crs == buurten.crs, "The points and the buurten use different CRSs"

# Use a spatial join to find the buurt that each point falls in
# (keeping just the buurtcode and geometry from the buurten dataframe)
points_in_buurten = gpd.sjoin(points_gdf, buurten[['buurtcode', 'geometry']], how='inner', predicate='within')

# Check that the join did not lose (or duplicate) any locations
assert len(points_in_buurten) == len(points_gdf), "The join lost or duplicated some locations"
print("All", len(points_in_buurten), "locations are in a buurt")

A quick map is a second check. Any locations in the water, or outside Amsterdam, would show that something has gone wrong:

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
buurten.plot(ax=ax, facecolor='none', edgecolor='grey', linewidth=0.5)
points_gdf.plot(ax=ax, markersize=0.5, color='darkblue', alpha=0.5)
ax.set_title("Photograph locations and buurt boundaries")
ax.set_axis_off()
plt.show()

### Where are the streets that match a description?

Each location has a score for each description, so we can map them. The scores are standardised, so a sequential colour map works well: higher scores (more like the description) are brighter.

<font color='orchid'> <b>Make a map of the <code>canal</code> score at each location, using <code>points_gdf</code>.</b></font> Plot the buurt boundaries underneath, as in the map above.

*Hints:* use `column='canal'`, `cmap='viridis'` and `legend=True`, and keep `markersize` small. With 30,000 points, some hide others. Sorting the points first, with `points_gdf.sort_values('canal')`, draws the highest scores on top.

Where are the locations that look most like canals? <font color='orchid'> <b>Then map one or two other descriptions.</b></font> Compare your maps with the SES map from Section 2: do any of the patterns look similar?

In [ ]:
# Add your code here



## 5. Predicting SES with a random forest

The 15 descriptions only use a small part of what CLIP 'sees'. The model in this section uses all 512 numbers in the embeddings instead. Each buurt needs one set of 512 numbers, so we use the **median embedding** of all the photographs in the buurt: for each of the 512 numbers, the median across the buurt's photographs.

### Calculating median embeddings

Before using the median embeddings for all of Amsterdam, you will calculate a few yourself, to see how they are made. The file `photo_embeddings_8_buurten.csv` contains the embeddings of the 1,560 photographs in the eight buurten from Section 3. There is one row per photograph, with:

- `point_id`: the location where the photograph was taken (the same IDs as in `points`)
- `heading`: the direction the photograph faces (0 = north, 90 = east, and so on)
- `emb_000` to `emb_511`: the 512 numbers in its embedding

In [ ]:
photo_embeddings = pd.read_csv(DATA_URL + "photo_embeddings_8_buurten.csv")

# A list of the names of the 512 embedding columns, which we will need several times
embedding_columns = [column for column in photo_embeddings.columns if column.startswith('emb_')]

print("photo_embeddings has", photo_embeddings.shape[0], "rows and", photo_embeddings.shape[1], "columns")
photo_embeddings.head()

We will do this in two steps.

<font color='orchid'> <b>Step 1: find the buurt of each photograph, by joining <code>photo_embeddings</code> to <code>points_in_buurten</code> on <code>point_id</code>. Then check with <code>assert</code> that no photographs were lost.</b></font>

*Hint:* you do not need another spatial join. The spatial join in Section 4 already found the buurt of every location, and it is stored in `points_in_buurten`. Use `merge()` as in Week 5, keeping only the two columns you need from `points_in_buurten`:

```python
photo_embeddings = photo_embeddings.merge(points_in_buurten[['point_id', 'buurtcode']], on='point_id', how='inner')
```

Afterwards there should still be 1,560 rows and there should be a new 'buurtcode' column

In [ ]:
# Add your code here



Does the table look as we would expect? (it should have a buurtcode as the final column)

In [ ]:
photo_embeddings.head()
#(or we could do `photo_embeddings.columns` just to see the columns)

Lets check how many points there are per buurt

In [ ]:
photo_embeddings['buurtcode'].value_counts()


<font color='orchid'> <b>Step 2: use <code>groupby()</code> to calculate the median of each of the 512 embedding columns in each buurt. Call the result <code>my_medians</code>.</b></font>

*Hint:* this is the same split-apply-combine pattern as in Week 3. Group by `'buurtcode'`, select the columns in `embedding_columns`, and use `.median()`. The result should have 8 rows (one per buurt) and 512 columns.

In [ ]:
# Add your code here



The research code calculated the median embeddings for all of Amsterdam in exactly this way. You could do the same for all 420 buurten, but you would need the embeddings of about 120,000 photographs. As a CSV file that would be about 500 MB: too big to share easily, and slow to load in Colab. Instead, they were calculated once, and we share the result (about 2 MB) along with the code that made it. Sharing the code and a small processed file like this is a common way to make a large analysis reproducible.

### Fitting the model

This uses the same steps as Week 7: split the data into training and test sets, fit a pipeline, and measure the error on the test data. First, load the median embeddings of all the buurten and join them to the SES scores:

In [ ]:
embeddings = pd.read_csv(DATA_URL + "amsterdam_buurt_clip_embeddings.csv")

model_data = embeddings.merge(ses, on='buurtcode', how='inner')

# To check the merged table looks OK. It should have ses columns at the end
print(model_data.columns)
model_data

Now we are ready to make our X and y data:

In [ ]:
assert len(model_data) == 420, "Expected 420 buurten with both embeddings and an SES score"

X = model_data[embedding_columns]  # The predictors: 512 numbers for each buurt
y = model_data['ses_score']        # What we want to predict

print("X has", X.shape[0], "rows and", X.shape[1], "columns")
print("y has", len(y), "rows (in one column)")

Now fit the model in three steps, using your Week 7 notebook as a guide. Use the variable names given in each step, because the code after Step 3 (and in Section 6) uses them.

<font color='orchid'> <b>Step 1: split <code>X</code> and <code>y</code> into training and test sets, with 20% of the buurten in the test set. Call the results <code>X_train</code>, <code>X_test</code>, <code>y_train</code> and <code>y_test</code>.</b></font>

*Hint:* use `train_test_split()` with `test_size=0.2` and `random_state=RANDOM_STATE`. The test set should have 84 buurten.

In [ ]:
# Add your code here



<font color='orchid'> <b>Step 2: make a pipeline called <code>rf_model</code> with a <code>StandardScaler</code> and a <code>RandomForestRegressor</code>, then fit it to the training data.</b></font>

*Hint:* name the two steps `'scaler'` and `'rf_regressor'`, as in Week 7. Give the random forest `random_state=RANDOM_STATE` as well, because a random forest uses random numbers too. Then use `rf_model.fit(X_train, y_train)`.

In [ ]:
# Add your code here



<font color='orchid'> <b>Step 3: use <code>rf_model</code> to predict the SES scores of the test buurten. Call the result <code>rf_predictions</code>. Then print the RMSE and R².</b></font>

*Hint:* use `rf_model.predict(X_test)`. Compare the predictions with `y_test` using `mean_squared_error()` (take the square root with `np.sqrt()` to get the RMSE) and `r2_score()`.

In [ ]:
# Add your code here



This chart shows how the predictions compare with the real SES scores of the test buurten:

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(y_test, rf_predictions, alpha=0.6)
ax.axline((0, 0), slope=1, linestyle='--', color='grey')  # A perfect model would put every point on this line
ax.set_xlabel("Actual SES score")
ax.set_ylabel("Predicted SES score")
ax.set_title("Random forest predictions for the test buurten")
plt.show()

How does this compare with the research? In Greater Manchester, a model trained on 1,702 neighbourhoods explained about 68% of the variation in deprivation (R² ≈ 0.68). Our R² is much lower. Possible reasons include:

- We have only 420 buurten, so the model has much less to learn from (and only 84 buurten in the test set).
- We used a random forest with its default settings. The research compared several models and tuned their settings (as in Week 7).
- SES-WOA measures something different from the Index of Multiple Deprivation, and Amsterdam's built environment is different from Manchester's.
- Some of the photographs are several years older than the SES scores (Section 2).

## 6. Same code, different answer?

In Week 7 you split the data with `train_test_split(X, y)` **without** a `random_state`. <font color='orchid'> <b>Run the next cell two or three times.</b></font> It is the same code, with the same data.

In [ ]:
# No random_state, so the split and the random forest are different every time
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
model = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('rf_regressor', RandomForestRegressor(n_jobs=-1))
])
model.fit(X_train, y_train)
print("R²:", round(r2_score(y_test, model.predict(X_test)), 3))

You get a different R² each time. This is a **repeatability** problem: you cannot repeat your own result, so nobody else can either. Setting the seed fixes it.

But a fixed seed does not make the result *right*. It just means you get the same answer every time. With a different seed you would get a different (equally valid) answer.

<font color='orchid'> <b>Hard task: write a <code>for</code> loop that fits the model with five different seeds (1, 2, 3, 4 and 5).</b></font> Use the seed for both `train_test_split()` and the random forest. Store each R² in a list called `r2_scores`, and then print the lowest, highest and mean R².

In [ ]:
# Add your code here



The R² depends a lot on which buurten happen to end up in the test set, because the test set is small. If you had run the model only once, you might have reported any of these numbers. This is why research papers use **cross-validation** (Week 7) and report the average, and why they should say which seeds they used.

## 7. Is your notebook reproducible?

### Step 1: Restart and run all

You have probably run some cells more than once and some cells out of order. Your notebook might only work because of something you ran earlier and later deleted. The only way to be sure is to start again from nothing.

<font color='orchid'> <b>In Colab, choose Runtime → Restart session and run all.</b></font> Then check:

- Did every cell run without an error?
- Is the R² in Section 5 the same as before? (It should be, because you set the seed.)
- Is the R² in the first cell of Section 6 the same as before? (It should not be, because there is no seed.)

If something goes wrong, fix it and restart again.

### Step 2: Save the notebook to GitHub

Choose **File → Save a copy in GitHub**, select your `GEOG5415` repository, and write a commit message that says what you did (for example, *"Complete week 9 practical"*).

### Step 3: Write a README

Every repository needs a `README.md` that tells people what the repository contains and how to use it.

Go to your repository on GitHub, click on `README.md`, then click the pencil icon to edit it. Copy the template below into it, fill in the parts in square brackets, and commit the changes.

```markdown
# GEOG5415M: [your name or student ID]

## What this repository contains
[One or two sentences about the repository. For example: "Notebooks from the GEOG5415M practicals.
The week 9 notebook asks whether street-view photographs can predict the socio-economic status
of neighbourhoods in Amsterdam."]

## Data
| Data | Source | Licence | Date downloaded |
|------|--------|---------|-----------------|
| Buurt boundaries and SES-WOA scores | Statistics Netherlands (CBS) | CC-BY 4.0 | [date] |
| Street-view photographs | Gemeente Amsterdam | CC-BY 4.0 | [date] |

[Say where the data files are, and how they were processed.]

## How to run the code
Open the notebook in Google Colab and choose Runtime → Run all.
It was last run with Python [version], pandas [version], geopandas [version] and scikit-learn [version].
```

---

## ✅ You have finished

**That is the end of the core practical.** You have:

- loaded and checked open data with `assert` statements
- used a foundation model to find and map streets that match a description
- checked that two spatial datasets line up before using them together
- calculated median embeddings for a few buurten yourself
- predicted socio-economic status with a random forest, and seen how much the result depends on the random seed
- checked that your notebook runs from start to finish, saved it to GitHub, and written a README

If you have time, try the optional extensions below. They are not needed for the assessment.

---

# Optional extensions

The extensions are independent, so you can do them in any order. Run all of the core sections first, because the extensions use the variables created there.

## Extension A: Which part of SES is most visible?

The SES score is made from three parts: `ses_wealth`, `ses_education` and `ses_labour` (employment history).

<font color='orchid'> <b>Repeat the loop that you wrote in Section 6 three times, predicting each part instead of <code>ses_score</code>.</b></font> Which part can the street-view model predict best? Can you think of a reason? (In Manchester, the education domain of the Index of Multiple Deprivation was predicted about as well as overall deprivation, even though education is not directly visible in a photograph.)

In [ ]:
# Add your code here



## Extension B: Where does the model go wrong?

With only 84 buurten in each test set it is hard to see where the model makes mistakes. `cross_val_predict()` gives every buurt a prediction from a model that did *not* see it during training: the data are split into five parts, and each part is predicted by a model trained on the other four.

<font color='orchid'> <b>Use <code>cross_val_predict()</code> to predict the SES score of every buurt, calculate the residuals (actual minus predicted), and map them.</b></font> Where does the model over-predict and under-predict? Look at the names of the buurten with the largest residuals.

In [ ]:
# Add your code here



## Extension C: Write your own description

To try your own descriptions you need CLIP itself. The cell below installs it and downloads the model (about 340 MB). This uses the same model, `ViT-B/32`, as the research. It runs on Colab's ordinary processor (CPU), so you do not need a GPU.

We do not have the embeddings of every photograph here (they are too large to share easily), so the function below compares your description with the **median embedding of each buurt** instead. This gives a score for each buurt: how closely its typical street matches your description.

In [ ]:
!pip install --quiet git+https://github.com/openai/CLIP.git

In [ ]:
import torch
import clip

clip_model, clip_preprocess = clip.load("ViT-B/32", device="cpu")

def description_scores(description):
    """Return a score for each buurt in model_data: how similar its median embedding is to the description."""
    with torch.no_grad():
        text_embedding = clip_model.encode_text(clip.tokenize([description])).float().numpy()[0]
    text_embedding = text_embedding / np.linalg.norm(text_embedding)
    buurt_embeddings = model_data[embedding_columns].to_numpy()
    buurt_embeddings = buurt_embeddings / np.linalg.norm(buurt_embeddings, axis=1, keepdims=True)
    return buurt_embeddings @ text_embedding  # cosine similarity

<font color='orchid'> <b>Write three descriptions of your own, calculate their scores with <code>description_scores()</code>, and find the correlation of each with <code>model_data['ses_score']</code>.</b></font> Start each one with *"a photo of"* (CLIP was trained on captions, and this usually helps). Can you find a description that is more strongly related to SES than any of the 15 in `prompts.csv`? (None of them has a correlation stronger than about ±0.26.)

In [ ]:
# Add your code here



## Extension D: Can you reproduce an embedding?

The folder `images_full_size/` contains four of the original photographs and the embeddings that the research code calculated for them, on a Mac laptop. Here you calculate the embedding of one photograph again, on a different computer, with the same model, and check whether you get the same numbers. (You need to run the first two cells of Extension C first.)

In [ ]:
stored = pd.read_csv(DATA_URL + "images_full_size/stored_clip_embeddings.csv")
stored_embedding = stored.loc[0, embedding_columns].to_numpy(dtype=float)
image = open_image(stored.loc[0, 'image_file'], folder="images_full_size/")

with torch.no_grad():
    new_embedding = clip_model.encode_image(clip_preprocess(image).unsqueeze(0)).float().numpy()[0]
new_embedding = new_embedding / np.linalg.norm(new_embedding)

print("Largest difference between any two numbers:", np.abs(new_embedding - stored_embedding).max())
print("Exactly the same (np.allclose with its default tolerance)?", np.allclose(new_embedding, stored_embedding))
print("The same to three decimal places (atol=0.001)?", np.allclose(new_embedding, stored_embedding, atol=0.001))

Are the embeddings identical? Why might the same model, given the same photograph, produce slightly different numbers on two different computers? Does a difference this small matter for the research?

*Something to think about:* different processors (and different versions of the libraries) do arithmetic in slightly different orders, and the tiny rounding errors add up. This is why reproducibility checks usually test whether results are the same *within a tolerance*, rather than exactly equal.